# SARF — One-Time Final Saudi Test Evaluation

**Owner:** A  
**Evaluation scope:** one frozen SVM pipeline, one frozen TextCNN checkpoint, and fifteen frozen AraBERT checkpoints (`E0`, `E1`, `E2`, `E3`, `EB` × seeds `42`, `123`, `2026`).

> This notebook has a strict two-stage procedure. The **preflight** verifies all 17 frozen artifacts, mappings, and MSA-validation evidence without opening Saudi test. The **official evaluation** opens Saudi test once, evaluates every model on that same held-out dataframe, writes all model-level artifacts, produces a single final comparison table, and locks the evaluation against accidental reruns.

Do not change any model, mapping, configuration, or preprocessing after the preflight passes.


## 1. Install the evaluation environment

Run this only for a new Colab session. If Colab asks for a restart after installation, restart and rerun the notebook from the top.


In [5]:
%pip install --upgrade --prefer-binary "tokenizers==0.21.4" "transformers==4.48.3" "accelerate==1.2.1" "datasets==3.2.0" "scikit-learn==1.6.1" "sentencepiece>=0.2.1"
%pip install --upgrade "PyArabic==0.6.15" "farasapy==0.1.1" "emoji==1.4.2"
%pip install --upgrade --no-deps "arabert==1.0.1"


## 2. Mount Drive and validate the evaluator source

This cell does not open model checkpoints or Saudi test.


In [6]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT")
EVALUATOR_PATH = PROJECT_ROOT / "src_04" / "sarf_final_saudi_evaluation.py"

assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"
assert EVALUATOR_PATH.is_file(), f"Unified evaluator not found: {EVALUATOR_PATH}"
assert torch.cuda.is_available(), "GPU is required for TextCNN and AraBERT final evaluation."

print({
    "status": "unified final evaluator source found",
    "project_root": str(PROJECT_ROOT),
    "evaluator": str(EVALUATOR_PATH),
    "gpu": torch.cuda.get_device_name(0),
    "saudi_test_accessed": False,
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'status': 'unified final evaluator source found', 'project_root': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT', 'evaluator': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/sarf_final_saudi_evaluation.py', 'gpu': 'Tesla T4', 'saudi_test_accessed': False}


## 3A. One-time recovery for the documented pre-test missing-path error

Run this cell **only if** the prior official command failed with `Missing Saudi test: .../saudi_test.csv` before evaluation began. It archives that path-only failure and removes its lock. It does not read the Saudi test, run models, produce predictions, or metrics. Then rerun the preflight cell before the official cell.


In [7]:
!python "{EVALUATOR_PATH}" --project-root "{PROJECT_ROOT}" --mode resolve_pretest_path_failure


{
  "owner": "A",
  "status": "recovered_pretest_path_failure",
  "recovery_at_utc": "2026-09-14T03:14:32.887378+00:00",
  "reason": "The previous official command stopped at a missing legacy file path before pd.read_csv, model inference, metrics, or output creation.",
  "previous_lock": {
    "owner": "A",
    "status": "failed",
    "started_at_utc": "2026-09-14T03:08:27.197458+00:00",
    "evaluation_scope": {
      "svm": 1,
      "textcnn": 1,
      "arabert": 15
    },
    "saudi_test_accessed": true,
    "failed_at_utc": "2026-09-14T03:08:27.538711+00:00",
    "failure_traceback": "Traceback (most recent call last):\n  File \"/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/sarf_final_saudi_evaluation.py\", line 653, in official\n    test_frame, test_metadata = read_held_out_test(paths, label_to_id)\n                                ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^\n  File \"/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/sarf_final_saudi_evaluation.py\", line 

## 3. Mandatory preflight — does not open Saudi test

This reads only frozen checkpoints, model configs, vocabulary, label mappings, and MSA-validation evidence. It verifies the SVM pipeline, the exact TextCNN architecture, all 15 AraBERT checkpoints, the common 77-label mapping, and that no final Saudi evaluation output already exists.


In [8]:
!python "{EVALUATOR_PATH}" --project-root "{PROJECT_ROOT}" --mode preflight


{
  "owner": "A",
  "status": "preflight_passed_no_saudi_test_access",
  "evaluation_scope": {
    "svm_models": 1,
    "textcnn_models": 1,
    "arabert_models": 15,
    "total_models": 17
  },
  "num_labels": 77,
  "arabert_config": "/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_config.json",
  "arabert_mapping": "/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_label_mapping.json",
  "arabert_checkpoints": [
    {
      "model_id": "AraBERT_E0_seed_42",
      "model_family": "AraBERTv2-base",
      "condition": "E0",
      "seed": 42,
      "checkpoint": "/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/06_models_and_checkpoints/arabert/E0/seed_42/best_checkpoint",
      "checkpoint_model_sha256": "7e91952fa4a7cb62cfb2d7fb78ffdb21078a3890294dd01ab37752c89a78c1eb",
      "checkpoint_config_sha256": "7565ae9db812e615c595410072058fdfdbd397ac4dc9205f7d398e8d546fbf4e",
      "msa_validation_metrics": {
        "macro_f1": 0.7349892966509619,
        

## 4. Official one-time held-out evaluation — run only after a successful preflight

This cell opens the Saudi test once and evaluates all 17 frozen models on the same dataframe. It writes model-level predictions, per-class metrics, confusion matrices, metrics JSON, seed-level results, and a final model-comparison CSV. It refuses to overwrite outputs or rerun after completion.

Before running, confirm that all final checkpoints are frozen and that no additional training or tuning will be performed.


In [9]:
FINAL_EVALUATION_APPROVAL = "OPEN_FINAL_HELD_OUT_SAUDI_TEST_FOR_ALL_17_FROZEN_MODELS"
assert FINAL_EVALUATION_APPROVAL == "OPEN_FINAL_HELD_OUT_SAUDI_TEST_FOR_ALL_17_FROZEN_MODELS"

!python "{EVALUATOR_PATH}" --project-root "{PROJECT_ROOT}" --mode official


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100% 241M/241M [07:18<00:00, 550kiB/s]
[2026-09-14 03:25:28,437 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
2026-09-14 03:25:45.538348: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
{
  "owner": "A",
  "status": "completed",
  "completed_at_utc": "2026-09-14T03:31:38.927025+00:00",
  "evaluation_type": "one_time_final_held_out_saudi_e

## 5. Display the final project comparison

Run only after the official cell reports `"status": "completed"`. This reads the completed aggregate CSV; it does not perform inference or reopen Saudi test.


In [10]:
import pandas as pd

comparison_path = PROJECT_ROOT / "07_results" / "final_saudi_evaluation" / "final_saudi_model_comparison.csv"
assert comparison_path.is_file(), "Final comparison table not found. Do not rerun the official evaluation cell."
comparison = pd.read_csv(comparison_path).sort_values("macro_f1_mean", ascending=False)
display(comparison)


,model,model_family,condition,seed_count,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,accuracy_mean,accuracy_std,msa_validation_macro_f1_mean,macro_f1_change_msa_to_saudi
0,AraBERT E0,AraBERTv2-base,E0,3,0.738918,0.004336,0.738779,0.005014,0.739944,0.004057,0.732088,0.006830
1,AraBERT EB,AraBERTv2-base,EB,3,0.732745,0.009626,0.732770,0.009528,0.734637,0.009460,0.730696,0.002049
2,AraBERT E3,AraBERTv2-base,E3,3,0.731622,0.009179,0.731458,0.009488,0.732961,0.009218,0.730242,0.001380
3,AraBERT E1,AraBERTv2-base,E1,3,0.730856,0.002163,0.731097,0.002390,0.732588,0.001817,0.733945,-0.003090
4,AraBERT E2,AraBERTv2-base,E2,3,0.728382,0.006157,0.728491,0.006767,0.730074,0.005888,0.729288,-0.000907
5,TF-IDF + LinearSVC,TF-IDF + LinearSVC,baseline,1,0.505079,0.000000,0.508930,0.000000,0.520112,0.000000,0.867210,-0.362131
6,TextCNN,TextCNN,baseline,1,0.355992,0.000000,0.359347,0.000000,0.349721,0.000000,0.814559,-0.458567
